# Preprocessing & Column Transformations
## Project: House Price Prediction using Linear Regression

This notebook demonstrates how we build the reusable data cleaning and preprocessing pipeline. We cover:
1. Handling missing values.
2. Standard scaling for numerical features.
3. Outlier identification and filtration using the IQR method.
4. Constructing and testing the Scikit-learn `Pipeline` and `ColumnTransformer` models.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

# Ensure project root in path
sys.path.append(str(Path(os.getcwd()).resolve().parent))
import config

### 1. Load Data
We load the training dataset using our flexible `DataLoader` class.

In [ ]:
from src.data_loader import DataLoader
loader = DataLoader()
df_raw = loader.load_data(config.TRAIN_PATH, is_training=True)
df_raw.head()

### 2. Outlier Removal via Interquartile Range (IQR)
We test outlier isolation on `sqft` and the target `price` columns to prevent high-leverage records from shifting our OLS regression line.

In [ ]:
from src.feature_engineering import remove_outliers_iqr

print(f"Rows before cleaning: {len(df_raw)}")
# Filter sqft outliers
df_filtered = remove_outliers_iqr(df_raw, "sqft", factor=2.0)
# Filter price outliers
df_filtered = remove_outliers_iqr(df_filtered, "price", factor=2.0)
print(f"Rows after outlier cleaning: {len(df_filtered)}")

### 3. Pipeline Transformation Construction
We build the modular preprocessing pipeline consisting of:
- `SimpleImputer` to handle missing values using the median strategy.
- `StandardScaler` to shift mean to 0 and scale variance to 1.

In [ ]:
from src.preprocessing import build_preprocessing_pipeline

# Split into features and target
X, y = loader.split_features_target(df_filtered)

# Instantiate pipeline
preprocessor = build_preprocessing_pipeline(config.NUMERICAL_FEATURES, config.CATEGORICAL_FEATURES)
print(preprocessor)

### 4. Test Preprocessing Output
We run `.fit_transform()` on our features (X) to verify that scaling and imputations are calculated correctly.

In [ ]:
X_transformed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# Convert back to DataFrame for validation
df_transformed = pd.DataFrame(X_transformed, columns=[name.split("__")[-1] for name in feature_names])
print("Transformed features sample:")
print(df_transformed.head())
print("\nFeatures Mean (Should be near 0):")
print(df_transformed.mean().round(4))
print("\nFeatures Std Dev (Should be near 1):")
print(df_transformed.std().round(4))

**Summary:**
- The preprocessing pipeline is fully functional and ready to be integrated with model training.
- Scaling has successfully transformed the features to mean 0 and standard deviation 1.
- This preprocessor transformer will be saved as part of our final model pipeline to ensure consistent data preparation during real-time web predictions.